In [2]:
"""
Knowledge Distillation — Script 2: Feature-Based
=================================================
Teacher : ResNet-50 (modified for CIFAR-10, pre-trained baseline)
Student : ResNet-18 (same CIFAR-10 modifications)
Dataset : CIFAR-10
Method  : Feature-based KD — match intermediate feature representations

WHAT IS FEATURE-BASED KD?
===========================
Beyond matching final outputs, the student is forced to mimic the teacher's
intermediate feature maps (activations at specific layers). This transfers
richer structural knowledge than response-based methods.

Three classic variants are implemented:

1. FitNets (Romero 2015)
   Match intermediate feature maps directly via a learned 1×1 conv adapter φ:
     L_hint = ||φ(F_teacher_l) - F_student_l||_F^2
   φ projects teacher channels → student channels (dimension adapter).

2. Attention Transfer / AT (Zagoruyko 2017)
   Match spatial attention maps (sum of squared feature values across channels):
     Q_l = sum_c(F_l,c^2)   (spatial attention map per layer l)
     L_AT = sum_l || Q_t_l / ||Q_t_l||_2 - Q_s_l / ||Q_s_l||_2 ||_2

3. NST — Neuron Selectivity Transfer (Huang 2017)
   Match the distribution of neuron activations using Maximum Mean Discrepancy (MMD)
   between teacher and student feature maps (polynomial kernel).

All variants add their feature loss on top of the standard CE loss:
  L_total = CE(y, p_student) + beta * L_feature

Sweeps:
  A — method       : [fitnet, attention, nst]          (beta=100, cosine, 10 ep)
  B — beta weight  : [10, 50, 100, 500, 1000]          (best method from A)
  C — hint layer   : [layer2, layer3, layer4]           (best method+beta, FitNet/AT only)

Output : __2__KD_feature_based.json
"""

import os, sys, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
TEACHER_MODEL_PATH    = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__2__KD_feature_based.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
KD_EPOCHS   = 10
USE_AMP     = (DEVICE.type == "cuda")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP: {USE_AMP}", flush=True)


# ── Model builders ─────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_resnet18_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_teacher(path: str) -> nn.Module:
    print(f"[load] Loading teacher from {path} ...", flush=True)
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval().to(DEVICE)
    print("[load] Teacher OK", flush=True)
    return model


# ── Channel dimensions per layer ───────────────────────────────────────────────
# ResNet-50: layer1=256, layer2=512, layer3=1024, layer4=2048
# ResNet-18: layer1=64,  layer2=128, layer3=256,  layer4=512
TEACHER_CHANNELS = {"layer1": 256, "layer2": 512, "layer3": 1024, "layer4": 2048}
STUDENT_CHANNELS = {"layer1": 64,  "layer2": 128, "layer3": 256,  "layer4": 512}


# ── Feature extractor wrapper ───────────────────────────────────────────────────
class FeatureExtractor(nn.Module):
    """
    Wraps a ResNet and returns (logits, {layer_name: feature_map}).
    Registers a forward hook on the named layers.
    """
    def __init__(self, model: nn.Module, layer_names: list):
        super().__init__()
        self.model       = model
        self.layer_names = layer_names
        self._features   = {}
        self._hooks      = []
        self._register_hooks()

    def _register_hooks(self):
        for name, module in self.model.named_modules():
            if name in self.layer_names:
                hook = module.register_forward_hook(
                    lambda m, inp, out, n=name: self._features.update({n: out})
                )
                self._hooks.append(hook)

    def forward(self, x):
        self._features = {}
        logits = self.model(x)
        return logits, {k: self._features[k] for k in self.layer_names
                        if k in self._features}

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


# ── Adapter module (FitNets) ────────────────────────────────────────────────────
class ChannelAdapter(nn.Module):
    """
    1×1 convolution projecting teacher channels → student channels.
    Required when C_teacher != C_student.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        nn.init.kaiming_normal_(self.proj.weight, mode="fan_out", nonlinearity="relu")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


# ── Feature-based loss functions ───────────────────────────────────────────────
def fitnet_loss(teacher_feat: torch.Tensor,
                student_feat: torch.Tensor,
                adapter: nn.Module) -> torch.Tensor:
    """
    FitNets: match projected teacher features to student features.
    L_hint = ||φ(F_teacher) - F_student||_F^2  (mean over elements)
    """
    projected = adapter(teacher_feat.detach())
    # Align spatial dimensions if they differ (avg pool to student size)
    if projected.shape[2:] != student_feat.shape[2:]:
        projected = F.adaptive_avg_pool2d(projected, student_feat.shape[2:])
    return F.mse_loss(student_feat, projected)


def attention_transfer_loss(teacher_feat: torch.Tensor,
                             student_feat: torch.Tensor) -> torch.Tensor:
    """
    Attention Transfer: match normalized spatial attention maps.
    Q = sum_c(F_c^2)  then normalize to unit L2 norm.
    L_AT = ||Q_t/||Q_t|| - Q_s/||Q_s||||_2
    """
    def attention_map(feat):
        # feat: (B, C, H, W) → (B, H*W)
        q = feat.pow(2).sum(dim=1)              # (B, H, W)
        q = q.view(q.size(0), -1)               # (B, H*W)
        return F.normalize(q, p=2, dim=1)

    # Align spatial size
    s_feat = student_feat
    t_feat = teacher_feat.detach()
    if t_feat.shape[2:] != s_feat.shape[2:]:
        t_feat = F.adaptive_avg_pool2d(t_feat, s_feat.shape[2:])

    at_t = attention_map(t_feat)
    at_s = attention_map(s_feat)
    return (at_t - at_s).pow(2).mean()


def nst_mmd_loss(teacher_feat: torch.Tensor,
                 student_feat: torch.Tensor,
                 num_samples: int = 512) -> torch.Tensor:
    """
    Neuron Selectivity Transfer via polynomial MMD kernel.
    Matches the distribution of neuron activations between teacher and student.

    For efficiency, we spatially pool features and use a random subset of
    channels. Polynomial kernel: k(x,y) = (x^T y / d + 1)^2
    """
    # Spatially global average pool → (B, C)
    t_pool = F.adaptive_avg_pool2d(teacher_feat.detach(), 1).squeeze(-1).squeeze(-1)
    s_pool = F.adaptive_avg_pool2d(student_feat, 1).squeeze(-1).squeeze(-1)

    # Project to same dimension if needed via random projection
    d = min(t_pool.size(1), s_pool.size(1), num_samples)
    if t_pool.size(1) != d:
        t_pool = t_pool[:, :d]
    if s_pool.size(1) != d:
        s_pool = s_pool[:, :d]

    # Normalize
    t_pool = F.normalize(t_pool, p=2, dim=1)
    s_pool = F.normalize(s_pool, p=2, dim=1)

    # Polynomial kernel MMD (degree=2):
    # MMD^2 = E[k(t,t)] - 2*E[k(t,s)] + E[k(s,s)]
    def poly_kernel(x, y):
        # (x^T y / d + 1)^2  averaged over batch
        gram = (x @ y.t() / d + 1).pow(2)
        return gram.mean()

    mmd = poly_kernel(t_pool, t_pool) - 2 * poly_kernel(t_pool, s_pool) + poly_kernel(s_pool, s_pool)
    return mmd.clamp(min=0)


# ── Data ────────────────────────────────────────────────────────────────────────
def get_train_loader():
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── Helpers ─────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device, non_blocking=True)
        # Handle FeatureExtractor wrapper
        out = model(inputs)
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_detailed(model: nn.Module, device: torch.device) -> dict:
    model = copy.deepcopy(model).eval().to(device)
    is_gpu = device.type == "cuda"

    # ── Single image ──────────────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)

    # ── Batch 128 ─────────────────────────────────────────────
    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }

def compute_flops(model: nn.Module) -> float:
    from thop import profile as thop_profile
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None


# ══════════════════════════════════════════════════════════════════════════════
# Core training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_feature_kd(
    teacher_raw: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    baseline_acc: float,
    teacher_size_mb: float,
    method: str      = "fitnet",     # "fitnet" | "attention" | "nst"
    hint_layer: str  = "layer3",     # which ResNet layer to use as hint
    beta: float      = 100.0,        # weight of feature loss
    lr: float        = 0.05,
    n_epochs: int    = KD_EPOCHS,
    sweep_name: str  = "method",
) -> dict:
    tag = f"{method}/{hint_layer}/beta={beta}"
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        # Fresh student
        student_raw = build_resnet18_cifar(NUM_CLASSES).to(DEVICE)

        # Wrap both in feature extractor
        teacher = FeatureExtractor(teacher_raw, [hint_layer]).to(DEVICE)
        student = FeatureExtractor(student_raw, [hint_layer]).to(DEVICE)
        teacher.eval()

        # Build channel adapter (only needed for FitNets)
        adapter = None
        if method == "fitnet":
            t_ch = TEACHER_CHANNELS[hint_layer]
            s_ch = STUDENT_CHANNELS[hint_layer]
            adapter = ChannelAdapter(t_ch, s_ch).to(DEVICE)
            print(f"      [adapter] {t_ch}→{s_ch} channels", flush=True)

        # Parameters: student + adapter (if any)
        params = list(student.parameters())
        if adapter is not None:
            params += list(adapter.parameters())

        optimizer = torch.optim.SGD(params, lr=lr,
                                    momentum=0.9, weight_decay=5e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc = []
        epoch_feat_loss = []
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            student.train()
            if adapter: adapter.train()
            correct = total = 0
            feat_loss_sum = 0.0
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    # Teacher forward (no grad)
                    with torch.no_grad():
                        t_logits, t_feats = teacher(inputs)

                    # Student forward
                    s_logits, s_feats = student(inputs)

                    # Task loss
                    ce_loss = F.cross_entropy(s_logits, targets)

                    # Feature matching loss
                    t_f = t_feats[hint_layer]
                    s_f = s_feats[hint_layer]

                    if method == "fitnet":
                        feat_loss = fitnet_loss(t_f, s_f, adapter)
                    elif method == "attention":
                        feat_loss = attention_transfer_loss(t_f, s_f)
                    else:  # nst
                        feat_loss = nst_mmd_loss(t_f, s_f)

                    loss = ce_loss + beta * feat_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (s_logits.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)
                feat_loss_sum += feat_loss.item() * targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done = batch_idx + 1
                    eta  = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  "
                          f"feat_loss={feat_loss_sum/total:.6f}  "
                          f"ETA={eta:.0f}s", flush=True)

            scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            epoch_feat_loss.append(round(feat_loss_sum / total, 8))
            sync()
            ep_time = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  feat_loss={feat_loss_sum/total:.6f}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start

        peak_gpu_mb = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                       if DEVICE.type == "cuda" else None)

        # Clean up hooks before evaluation
        teacher.remove_hooks()
        student.remove_hooks()

        print("      [eval] Evaluating student ...", flush=True)
        metrics  = evaluate(student.model, test_loader, DEVICE)
        inf      = measure_inference_detailed(student.model, DEVICE)
        size_mb  = model_size_mb(student.model)
        acc_drop = baseline_acc - metrics["accuracy"]
        flops_G  = compute_flops(student.model)
        total_params = param_count(student.model)

        # ── Save model ────────────────────────────────────────────
        save_dir = "__2__saved_models_KD"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/student_resnet18_{method}_{hint_layer}_beta{int(beta)}.pth"
        torch.save(student.model.state_dict(), save_path)
        print(f"      [{tag}] ✓ Saved → {save_path}", flush=True)

        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Size={size_mb:.2f}MB  Single={inf['single_img_gpu_ms']:.2f}ms  "
              f"Throughput={inf['throughput_imgs_sec']:.0f}imgs/s  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            # identification
            "sweep"             : sweep_name,
            "method"            : method,
            "hint_layer"        : hint_layer,
            "beta"              : beta,
            "lr"                : lr,
            "lr_schedule"       : "cosine",
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            # training stats
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "epoch_feat_loss"   : epoch_feat_loss,
            "peak_gpu_memory_mb": peak_gpu_mb,
            # ── Standardised metrics block ────────────────────────
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "params"            : total_params,
            "params_nonzero"    : total_params,   # feature KD = dense model
            "sparsity_actual"   : 0.0,
            "size_mb"           : round(size_mb, 4),
            "flops_G"           : flops_G,
            "inference_ms"      : inf,
            # size comparisons
            "teacher_size_mb"   : round(teacher_size_mb, 4),
            "compression_ratio" : round(teacher_size_mb / size_mb, 4) if size_mb else None,
            # adapter info
            "adapter_params"    : param_count(adapter) if adapter else 0,
            # meta
            "saved_model_path"  : save_path,
            "description"       : (
                f"Feature-Based KD ({method}, layer={hint_layer}, "
                f"beta={beta}, cosine, epochs={n_epochs})"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "sweep": sweep_name, "method": method,
            "hint_layer": hint_layer, "beta": beta,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ────────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Feature-Based Knowledge Distillation")
    print("  Teacher: ResNet-50 (modified)  →  Student: ResNet-18 (modified)")
    print(f"  Device: {DEVICE}  |  Epochs: {KD_EPOCHS}  |  Batch: {BATCH_SIZE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline teacher acc: {baseline_acc:.4f}\n", flush=True)

    teacher = load_teacher(TEACHER_MODEL_PATH)
    teacher_size_mb = model_size_mb(teacher)
    student_ref     = build_resnet18_cifar(NUM_CLASSES)
    student_size_mb = model_size_mb(student_ref)

    print(f"  Teacher — Size: {teacher_size_mb:.2f} MB  Params: {param_count(teacher):,}")
    print(f"  Student — Size: {student_size_mb:.2f} MB  Params: {param_count(student_ref):,}\n")

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    results = []

    # ── Sweep A: Method (fitnet / attention / nst) ────────────────────────────
    print("  ── Sweep A: Method  (layer3, beta=100) ───────────────────────────", flush=True)
    sweep_a = []
    for method in ["fitnet", "attention", "nst"]:
        row = run_feature_kd(
            teacher, train_loader, test_loader, baseline_acc, teacher_size_mb,
            method=method, hint_layer="layer3", beta=100.0,
            lr=0.05, n_epochs=KD_EPOCHS, sweep_name="A_method",
        )
        results.append(row)
        sweep_a.append(row)

    valid_a = [r for r in sweep_a if r.get("accuracy") is not None]
    if not valid_a:
        print("  ✗ Sweep A failed entirely.", flush=True)
        return
    best_method = max(valid_a, key=lambda r: r["accuracy"])["method"]
    print(f"\n  Best method: {best_method}", flush=True)

    # ── Sweep B: Beta weight ──────────────────────────────────────────────────
    print(f"\n  ── Sweep B: Beta weight  (method={best_method}, layer3) ──────────", flush=True)
    sweep_b = []
    for beta in [10.0, 50.0, 100.0, 500.0, 1000.0]:
        row = run_feature_kd(
            teacher, train_loader, test_loader, baseline_acc, teacher_size_mb,
            method=best_method, hint_layer="layer3", beta=beta,
            lr=0.05, n_epochs=KD_EPOCHS, sweep_name="B_beta",
        )
        results.append(row)
        sweep_b.append(row)

    valid_b = [r for r in sweep_b if r.get("accuracy") is not None]
    if valid_b:
        best_beta = max(valid_b, key=lambda r: r["accuracy"])["beta"]
        print(f"\n  Best beta: {best_beta}", flush=True)
    else:
        best_beta = 100.0

    # ── Sweep C: Hint layer (skip for nst — not as sensitive to layer choice) ─
    if best_method != "nst":
        print(f"\n  ── Sweep C: Hint layer  (method={best_method}, beta={best_beta}) ──", flush=True)
        for layer in ["layer2", "layer3", "layer4"]:
            row = run_feature_kd(
                teacher, train_loader, test_loader, baseline_acc, teacher_size_mb,
                method=best_method, hint_layer=layer, beta=best_beta,
                lr=0.05, n_epochs=KD_EPOCHS, sweep_name="C_hint_layer",
            )
            results.append(row)
    else:
        print(f"\n  ── Sweep C: Skipped for NST (layer choice less impactful) ───", flush=True)

    # ── Best overall ──────────────────────────────────────────────────────────
    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("  ✗ No valid results.", flush=True)
        return
    best = max(valid, key=lambda r: r["accuracy"])

    teacher_inf = measure_inference_detailed(teacher, DEVICE)

    report = {
        "method"       : "feature_based_kd",
        "description"  : "Feature-Based Knowledge Distillation — intermediate layer matching",
        "teacher_arch" : "ResNet-50 (CIFAR-10 modified)",
        "student_arch" : "ResNet-18 (CIFAR-10 modified)",
        "dataset"      : "CIFAR-10",
        "train_device" : str(DEVICE),
        "use_amp"      : USE_AMP,
        "kd_epochs"    : KD_EPOCHS,
        "baseline"     : baseline,
        "teacher_info" : {
            "size_mb"     : round(teacher_size_mb, 4),
            "inference_ms": teacher_inf,
            "params"      : param_count(teacher),
        },
        "student_info" : {
            "size_mb"     : round(student_size_mb, 4),
            "params"      : param_count(student_ref),
        },
        "best_config"  : {
            "method"          : best["method"],
            "hint_layer"      : best["hint_layer"],
            "beta"            : best["beta"],
            "accuracy"        : best["accuracy"],
            "accuracy_drop"   : best["accuracy_drop"],
            "size_mb"         : best["size_mb"],
            "flops_G"         : best.get("flops_G"),
            "inference_ms"    : best.get("inference_ms"),
            "compression_ratio": best.get("compression_ratio"),
            "saved_model_path": best.get("saved_model_path"),
        },
        "sweeps"       : {
            "A_method"    : "fitnet vs attention vs nst  (layer3, beta=100)",
            "B_beta"      : "beta in [10,50,100,500,1000]  (best method, layer3)",
            "C_hint_layer": "layer in [layer2,layer3,layer4]  (best method+beta)",
        },
        "results"      : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Best: method={best['method']} | layer={best['hint_layer']} | "
          f"beta={best['beta']} | Acc={best['accuracy']:.4f} | "
          f"Drop={best['accuracy_drop']:+.4f}")
    if best.get("inference_ms"):
        inf = best["inference_ms"]
        print(f"  Single={inf['single_img_gpu_ms']:.2f}ms  "
              f"Throughput={inf['throughput_imgs_sec']:.0f}imgs/s  "
              f"Ratio={best.get('compression_ratio', 'N/A')}×")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] AMP: True

  Feature-Based Knowledge Distillation
  Teacher: ResNet-50 (modified)  →  Student: ResNet-18 (modified)
  Device: cuda  |  Epochs: 10  |  Batch: 128

  Baseline teacher acc: 0.9320

[load] Loading teacher from ../__2__baseline_resnet50_cifar10.pth ...
[load] Teacher OK
  Teacher — Size: 90.03 MB  Params: 23,520,842
  Student — Size: 42.70 MB  Params: 11,173,962

  ── Sweep A: Method  (layer3, beta=100) ───────────────────────────

  ┌─ [fitnet/layer3/beta=100.0]
      [adapter] 1024→256 channels
      [epoch 1/10] batch 100/391  acc=0.2091  feat_loss=0.083101  ETA=23s
      [epoch 1/10] batch 200/391  acc=0.2655  feat_loss=0.044749  ETA=14s
      [epoch 1/10] batch 300/391  acc=0.3011  feat_loss=0.031280  ETA=7s
      [epoch 1/10] DONE  acc=0.3245  feat_loss=0.024789  time=27.8s  ETA_remaining=250s
      [epoch 2/10] batch 100/391  acc=0.4248  feat_loss=0